In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.table("bikesales.bronze.addresses")

In [0]:
display(df)

In [0]:
df.groupBy("ADDRESSID") \
    .count() \
    .filter(col("count") > 1) \
    .display()

In [0]:
df.filter(col("CITY").isNull()).display()

In [0]:
df.filter(col("POSTALCODE").isNull()).display()

In [0]:
df.filter(col("STREET").isNull()).display()

In [0]:
df.select("CITY", "POSTALCODE", "STREET", "BUILDING").filter(col("BUILDING").isNull()).display()

In [0]:
df.filter(col("COUNTRY").isNull()).display()

In [0]:
df.select("COUNTRY").distinct().display()

In [0]:
df.select("REGION").distinct().display()

In [0]:
df.select("ADDRESSTYPE").distinct().display()

In [0]:
display(df)

In [0]:
# Adicionar coluna IS_CURRENT antes da conversão (usando valor integer)
df = df.withColumn(
    "IS_CURRENT",
    when(col("VALIDITY_ENDDATE") == 99991231, True).otherwise(False)
)

# Converter colunas de validade de integer para date
df = df.withColumn(
    "VALIDITY_STARTDATE", 
    to_date(col("VALIDITY_STARTDATE").cast("string"), "yyyyMMdd")
).withColumn(
    "VALIDITY_ENDDATE", 
    to_date(col("VALIDITY_ENDDATE").cast("string"), "yyyyMMdd")
)

In [0]:
df.select("ADDRESSID", "CITY", "VALIDITY_STARTDATE", "VALIDITY_ENDDATE", "IS_CURRENT").limit(5).display()

In [0]:
df.write.mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable("bikesales.silver.addresses")